# 04.2 · Data leakage: the mistake that makes every model look great

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/04-trustworthy-models/02-data-leakage/lesson.ipynb)

Read the [README](README.md) first. This notebook rebuilds the three kinds of leakage on ShopWave data, shows each one producing a score that is too good to be true, and then fixes it:

1. **Target leakage**: a feature that already contains the answer.
2. **Train–test contamination**: preparing data with the test rows included.
3. **Duplicates across the split**: the same row on both sides.

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import sys, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail()                                   # ~1 s after the first run
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

def customer_table(cutoff, months=3):
    """One row per customer: features from before `cutoff`, churned = no order in the next `months`."""
    cutoff = pd.Timestamp(cutoff)
    past = sales[sales["InvoiceDate"] < cutoff]
    future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < cutoff + pd.DateOffset(months=months))]
    c = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                        products=("StockCode", "nunique"), country=("Country", "first"),
                                        first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
    c["tenure_days"] = (cutoff - c["first_order"]).dt.days
    c["days_since_last"] = (cutoff - c["last_order"]).dt.days
    came_back = future.groupby("Customer ID")["Revenue"].sum().reindex(c.index).fillna(0) > 0
    c["churned"] = (~came_back).astype(int)
    return c.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
cust = customer_table("2011-09-01")
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}")

3,317 customers, churn rate 43.6%


## 1. Target leakage: the 97% churn model

This is the model that broke in production. Its author added one column from the database export: `recency_days`, the number of days since each customer's last order, **measured on the day the data was exported** (9 December 2011).

In [2]:
# the column the old export "helpfully" added: days since last order, measured on the EXPORT date
export_date = sales["InvoiceDate"].max()
last_ever = sales.groupby("Customer ID")["InvoiceDate"].max()
cust["recency_days"] = (export_date - last_ever).dt.days.reindex(cust.index)
print("export date:", export_date.date())
cust[["days_since_last", "recency_days", "churned"]].head()

export date: 2011-12-09


,days_since_last,recency_days,churned
Customer ID,,,
12346.0,225,325,1
12347.0,29,1,0
12348.0,148,74,0
12350.0,210,309,1
12352.0,162,35,0


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

leaky = features + ["recency_days"]
X_tr, X_te, y_tr, y_te = train_test_split(cust[leaky], cust["churned"], test_size=0.25,
                                          random_state=42, stratify=cust["churned"])
model = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr, y_tr)
print(f"test accuracy with recency_days: {model.score(X_te, y_te):.1%}")
print(pd.Series(model.feature_importances_, index=leaky).round(2).to_dict())

test accuracy with recency_days: 97.6%
{'orders': 0.0, 'spend': 0.02, 'products': 0.0, 'tenure_days': 0.0, 'days_since_last': 0.0, 'recency_days': 0.98}


A **97.6%** churn model, with `recency_days` doing 98% of the work. Here is why, by the logic in the README: the target window is 1 September to 1 December, and the export happened on 9 December, 99 days after the cut-off. Anyone who bought in the window has `recency_days` below 99. Anyone who did not has at least 99, unless they happened to buy in the eight days between 1 and 9 December. The column *is* the answer.

In [4]:
pd.crosstab(cust["recency_days"] >= 99, cust["churned"],
            rownames=["recency_days >= 99"], colnames=["churned"])

churned,0,1
recency_days >= 99,,
False,1870,81
True,1,1365


One rule, "`recency_days` ≥ 99 means churned", gets 1,365 + 1,870 of 3,317 customers right (97.5%). The 81 exceptions are churners who came back in early December, after the target window closed.

**Going live.** On 1 September next year, ShopWave exports the data *that day*. On that day `recency_days` equals `days_since_last`, because both are measured on the same date. The magic column has no information about the future any more. Let us simulate it:

In [5]:
live = X_te.copy()
live["recency_days"] = live["days_since_last"]       # what the column contains on the scoring day
print(f"notebook (test set):  {model.score(X_te, y_te):.1%}")
print(f"live (same customers): {model.score(live, y_te):.1%}")

notebook (test set):  97.6%
live (same customers): 63.1%


From 97.6% to 63.1%. That is the 90% → 60% story from the module introduction.

**The fix:** every feature must be computable *at the moment of prediction*. Drop `recency_days`; use `days_since_last`, which is measured at the cut-off.

In [6]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"train {len(X_train):,} / test {len(X_test):,}")

fixed = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)
print(f"honest model, test accuracy: {fixed.score(X_test, y_test):.1%}")

train 2,487 / test 830
honest model, test accuracy: 65.1%


A boring 65%, and a true one.

## 2. Train–test contamination: learning from the exam paper

Any step that *looks at the data to decide something* (the mean for scaling, the median for filling gaps, which features to keep) must look only at training rows. Here is the most dramatic version: feature selection on **pure noise**.

In [7]:
rng = np.random.default_rng(0)
X_noise = rng.normal(size=(200, 2000))     # 200 customers, 2,000 columns of pure noise
y_noise = rng.integers(0, 2, 200)          # labels from a coin flip

from sklearn.linear_model import LogisticRegression

# WRONG: pick the 20 columns most correlated with y using ALL rows, then split
corr = np.abs([np.corrcoef(X_noise[:, j], y_noise)[0, 1] for j in range(2000)])
top20 = np.argsort(corr)[-20:]
Xa, Xb, ya, yb = train_test_split(X_noise[:, top20], y_noise, test_size=0.3, random_state=0)
print(f"select, then split: {LogisticRegression().fit(Xa, ya).score(Xb, yb):.0%} on coin-flip labels")

select, then split: 75% on coin-flip labels


In [8]:
# RIGHT: split first, then pick the 20 columns using the training rows only
Xa, Xb, ya, yb = train_test_split(X_noise, y_noise, test_size=0.3, random_state=0)
corr_tr = np.abs([np.corrcoef(Xa[:, j], ya)[0, 1] for j in range(2000)])
top20 = np.argsort(corr_tr)[-20:]
print(f"split, then select: {LogisticRegression().fit(Xa[:, top20], ya).score(Xb[:, top20], yb):.0%}")

split, then select: 45%


75% accuracy predicting coin flips, from nothing but a wrong order of operations. With 2,000 random columns, some will match the 200 labels by chance; choosing them with the test labels in view bakes that chance into the "model". Done in the right order, the score collapses to what it should be, around 50% (45% here, by luck).

**Scaling before splitting** is the same mistake in a milder form. On the churn data the damage is small:

In [9]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

scaled_all = StandardScaler().fit_transform(cust[features])          # WRONG: test rows set the mean/std
Xa, Xb, ya, yb = train_test_split(scaled_all, cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"scaled before split: {KNeighborsClassifier(15).fit(Xa, ya).score(Xb, yb):.1%}")

sc = StandardScaler().fit(X_train)                                    # RIGHT: train rows only
print(f"scaled after split:  {KNeighborsClassifier(15).fit(sc.transform(X_train), y_train).score(sc.transform(X_test), y_test):.1%}")

scaled before split: 63.6%
scaled after split:  64.5%


Less than a point apart, and here the wrong way even scores a little *lower*. Small, but it is still the wrong habit: on smaller data, rarer classes or more aggressive steps (feature selection, target encoding in 04.4) the same mistake becomes the 75% above. Lesson 04.8 makes the right order automatic with a `Pipeline`.

## 3. Duplicates across the split

Suppose the export job ran twice for some customers, so 30% of rows appear two times. A random split then puts many rows in the test set whose identical twin is in the training set.

In [10]:
doubled = pd.concat([cust, cust.sample(frac=0.3, random_state=1)])
print(f"{len(doubled):,} rows for {doubled.index.nunique():,} customers")

Xa, Xb, ya, yb = train_test_split(doubled[features], doubled["churned"], test_size=0.25, random_state=0)
twins = Xb.index.isin(Xa.index).mean()
deep = DecisionTreeClassifier(random_state=0).fit(Xa, ya)
print(f"test rows whose customer is also in train: {twins:.0%}")
print(f"unlimited tree, duplicated data: {deep.score(Xb, yb):.1%}")

4,312 rows for 3,317 customers
test rows whose customer is also in train: 35%
unlimited tree, duplicated data: 71.6%


In [11]:
Xa, Xb, ya, yb = train_test_split(cust[features], cust["churned"], test_size=0.25, random_state=0)
print(f"unlimited tree, de-duplicated:   {DecisionTreeClassifier(random_state=0).fit(Xa, ya).score(Xb, yb):.1%}")

unlimited tree, de-duplicated:   61.1%


Ten points of fake accuracy from memorised twins. The fix is either to **de-duplicate** before splitting, or, when one customer legitimately has several rows, to split **by customer** with `GroupShuffleSplit` / `GroupKFold` so all of a customer's rows land on the same side.

In [12]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(test_size=0.25, random_state=0)
tr, te = next(gss.split(doubled, groups=doubled.index))
print("customers on both sides after a group split:",
      len(set(doubled.index[tr]) & set(doubled.index[te])))

customers on both sides after a group split: 0


## Quiz answers

1. Target leakage: a feature contains information that would not be available at prediction time, often because it was computed after the target happened.
2. `recency_days` was measured on the export date, after the target window, so it encoded whether the customer came back. On the scoring day it equals `days_since_last` and carries no future information, so live accuracy dropped to 63.1%.
3. Fitting any data-dependent step (scaler, imputer, feature selector, encoder) on all rows before splitting, so the test rows influence training.
4. The test set contains copies of training rows; a flexible model memorises them and appears accurate on "unseen" data that it has, in fact, seen.
5. Be suspicious: check feature importances for one dominant feature, ask *when* each feature is known, and compare with a simple baseline. Then test on a later time period.

**Next:** [04.3 Missing values and outliers](../03-missing-and-outliers/)